In [ ]:
import os
import zipfile
import copy
import shutil
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, models, transforms

# =========================
# MONTAJE Y DESCOMPRESIÓN
# =========================
from google.colab import drive
drive.mount('/content/drive')

ZIP_PATH = "/content/drive/MyDrive/Fruit-Images-Dataset-master.zip"
EXTRACT_PATH = "dataset_frutas"

if os.path.exists(ZIP_PATH):
    print(f"Descomprimiendo {ZIP_PATH}...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_PATH)
    print("Dataset descomprimido con éxito.")
else:
    print(f"Error crítico: No se encuentra {ZIP_PATH}. Revisa el nombre del archivo o la subida a Drive.")
    exit()

# =========================
# CONFIGURACIÓN
# =========================
# Verifica si el ZIP incluye la carpeta padre en su estructura interna
BASE_PATH = os.path.join(EXTRACT_PATH, "Fruit-Images-Dataset-master")
if not os.path.exists(os.path.join(BASE_PATH, "Training")):
    BASE_PATH = EXTRACT_PATH

TRAIN_DIR = os.path.join(BASE_PATH, "Training")
TEST_DIR = os.path.join(BASE_PATH, "Test")
OUTPUT_DIR = "modelo_final"
os.makedirs(OUTPUT_DIR, exist_ok=True)

IMG_SIZE = 224
BATCH_SIZE = 64
EPOCHS = 10
LEARNING_RATE = 0.001
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Dispositivo de entrenamiento: {DEVICE}")

# =========================
# TRANSFORMACIONES
# =========================
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# =========================
# CARGA DE DATOS
# =========================
full_train_dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=train_transform)
test_dataset = datasets.ImageFolder(root=TEST_DIR, transform=test_transform)

class_names = full_train_dataset.classes
num_classes = len(class_names)

# Generar labels.txt
with open(os.path.join(OUTPUT_DIR, "labels.txt"), "w") as f:
    for name in class_names:
        f.write(name + "\n")

# Split validación
val_size = int(0.15 * len(full_train_dataset))
train_size = len(full_train_dataset) - val_size
train_ds, val_ds = random_split(full_train_dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# =========================
# MODELO RESNET18
# =========================
try:
    weights = models.ResNet18_Weights.IMAGENET1K_V1
    model = models.resnet18(weights=weights)
except AttributeError:
    model = models.resnet18(pretrained=True)

for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=LEARNING_RATE)

# =========================
# ENTRENAMIENTO
# =========================
best_acc = 0.0
best_model_wts = copy.deepcopy(model.state_dict())

for epoch in range(EPOCHS):
    model.train()
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

    model.eval()
    corrects = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            corrects += torch.sum(preds == labels.data)

    val_acc = corrects.double() / val_size
    print(f"Epoch {epoch+1}/{EPOCHS} - Val Acc: {val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        best_model_wts = copy.deepcopy(model.state_dict())

# =========================
# EXPORTACIÓN ONNX
# =========================
# Install onnxscript if not already installed
!pip install onnxscript

model.load_state_dict(best_model_wts)
model.eval()
dummy_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
onnx_path = os.path.join(OUTPUT_DIR, "fruits_resnet18.onnx")

torch.onnx.export(
    model, dummy_input, onnx_path,
    input_names=['input_0'], output_names=['output_0'],
    opset_version=18 # Changed opset_version to 18
)
print(f"Exportación temporal ONNX exitosa en: {onnx_path}")

# =========================
# GUARDADO EN GOOGLE DRIVE
# =========================
DRIVE_OUTPUT = "/content/drive/MyDrive/modelo_frutas_exportado"
shutil.copytree(OUTPUT_DIR, DRIVE_OUTPUT, dirs_exist_ok=True)
print(f"Archivos transferidos permanentemente a tu Drive: {DRIVE_OUTPUT}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Descomprimiendo /content/drive/MyDrive/Fruit-Images-Dataset-master.zip...
Dataset descomprimido con éxito.
Dispositivo de entrenamiento: cuda
Epoch 1/10 - Val Acc: 0.9984
Epoch 2/10 - Val Acc: 0.9996
Epoch 3/10 - Val Acc: 0.9994
Epoch 4/10 - Val Acc: 1.0000
Epoch 5/10 - Val Acc: 0.9999
Epoch 6/10 - Val Acc: 0.9999
Epoch 7/10 - Val Acc: 0.9999
Epoch 8/10 - Val Acc: 0.9999
Epoch 9/10 - Val Acc: 0.9999
Epoch 10/10 - Val Acc: 0.9999


W0509 09:44:30.882000 2857 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0509 09:44:30.885000 2857 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'rois' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0509 09:44:30.887000 2857 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
W0509 09:44:30.888000 2857 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.


[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Exportación temporal ONNX exitosa en: modelo_final/fruits_resnet18.onnx
Archivos transferidos permanentemente a tu Drive: /content/drive/MyDrive/modelo_frutas_exportado


In [ ]:
import os
import torch
import shutil

print("Forzando exportación a ONNX Opset 18...")

# Asegurarnos de que el modelo está en modo evaluación
model.eval()

# Crear un tensor de prueba
dummy_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
onnx_path_fixed = os.path.join(OUTPUT_DIR, "fruits_resnet18_fixed.onnx")

# Exportar el modelo directamente (sin trazar con torch.jit.trace)
torch.onnx.export(
    model, # Export the original model, not the traced one
    dummy_input,
    onnx_path_fixed,
    input_names=['input_0'],
    output_names=['output_0'],
    opset_version=18 # Changed opset_version to 18 to align with system preference
)

# 3. Copiar el archivo corregido a tu Google Drive
DRIVE_FIXED_PATH = "/content/drive/MyDrive/modelo_frutas_exportado/fruits_resnet18_fixed.onnx"
shutil.copy(onnx_path_fixed, DRIVE_FIXED_PATH)

print(f"¡Éxito! Modelo corregido y guardado en tu Drive en: {DRIVE_FIXED_PATH}")

W0509 09:44:33.940000 2857 torch/onnx/_internal/exporter/_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 11 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


Forzando exportación a ONNX Opset 11...


W0509 09:44:35.051000 2857 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0509 09:44:35.058000 2857 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'rois' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0509 09:44:35.061000 2857 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
W0509 09:44:35.064000 2857 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.


[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_str, target_version)
                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: /github/workspace/onnx/version_converter/adapters/axes_input_to_attribute.h:68: adapt: Asserti

¡Éxito! Modelo corregido y guardado en tu Drive en: /content/drive/MyDrive/modelo_frutas_exportado/fruits_resnet18_fixed.onnx


In [ ]:
import os
import torch

# Aseguramos la ruta en tu Drive
DRIVE_OUTPUT = "/content/drive/MyDrive/modelo_frutas_exportado"
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

# Ruta del archivo de pesos
pth_path = os.path.join(DRIVE_OUTPUT, "frutas_resnet18_pesos.pth")

# Guardamos el cerebro del modelo
torch.save(model.state_dict(), pth_path)

print(f"¡SALVADO! Los pesos se han guardado de forma segura en: {pth_path}")

¡SALVADO! Los pesos se han guardado de forma segura en: /content/drive/MyDrive/modelo_frutas_exportado/frutas_resnet18_pesos.pth
